## DFT vs. MLIPs Energy Comparison Pipeline

This notebook compares MLIP-predicted energies and forces against DFT reference calculations. For each frame in a DFT trajectory, the MLIP takes the DFT-computed structure (atomic positions, cell) as input and performs a single-point evaluation, predicting E and F(r) for that fixed, pre-relaxed geometry. 

The question this tries to answer is: *Given the same structure DFT saw, how close are the MLIP energies and forces?*

Comparison plots are then generated for raw potential energy, relative energy along the trajectory, and adsorption energy (E_ads = E_tot − E_slab − E_gas).

Currently, models used are MACE, MatterSim, and UMA (each run by a script in `model-scripts`).

Note: Although the MLIPs models below are not trained on magnetic field data, magnetically-perturbed DFT data was given and used when constructing the pipeline.

**Future Work**

* [ ] Make parameters/code that handles magnetic data optional (so the pipeline works with non-magnetized DFT runs)
* [√] Add some I/O handling for the `# DON'T RUN ME` cells so when you press "Run All", the program stops before those cells and asks the user if they'd like to run or skip them (check out the ipywidgets library for this)
* [ ] Address some of the other misc. feedback given on Wednesday's meeting (see notes below)


In [2]:
# TODO: write notebook description

# 12 plots instead of 4 (separate out models)
# try MAE by subtracting out different model/dft baselines and running MAE on that
# to consider: when doing E_adsorption, make sure E_total is on the same scale, or it might not work well?

#### 0. Configuration

In [4]:
from pathlib import Path
import numpy as np
from ase.io import iread, read, write

# wait for user input before running cells with expensive calculations
def confirm(prompt="Run long + expensive calculation?"):
    return input(prompt + " [y/n]: ").strip().lower() == 'y'

In [5]:
# builds a dict describing 1 DFT adsorbate run
# ads: single adsorbate string or two joined like ads1_ads2
# template: path template string with {ads}, {slab}, {config_id} placeholders;
#           define at call site (TEMPLATE in next cell) to handle
#           directory structure variants.

def make_run(ads, slab, config_id, template):
    return {
        "ads":       ads,
        "slab":      slab,
        "config_id": config_id,  # use arbitrary directory numbering to label ads config for now
        "path":      template.format(ads=ads, slab=slab, config_id=config_id),
    }

In [6]:
# directory path template for adsorbate systems on YPdCu slab
YPDCU_TEMPLATE = "dft-data/YPdCu/{ads}_on_{slab}/{ads}_{slab}_{config_id}/vasprun.xml"
CH3CHO_YPDCU_TEMPLATE = "dft-data/YPdCu/{ads}_{slab}_{config_id}/vasprun.xml"

RUNS = (
    [make_run("CH3CHO",       "YO4PdCu", 3, CH3CHO_YPDCU_TEMPLATE)] +
    [make_run("CO_CHO",  "YO4PdCu", i, YPDCU_TEMPLATE) for i in range(1, 15)] +
    [make_run("COCHO",        "YO4PdCu", i, YPDCU_TEMPLATE) for i in range(1, 19)] +
    [make_run("COOH",         "YO4PdCu", i, YPDCU_TEMPLATE) for i in range(1, 16)]
)

#### 1. DFT data ingestion 
(extract all frames from DFT runs)

In [7]:
# load vasp trajectories into ASE Atoms obj.

# for trajectory files with multiple steps
def load_frames(path):
    frames = []
    for f in iread(path):
        frames.append(f)
    return frames

# for single-structure files (references, loads last by default)
def load_frame(path, index=-1):
    frame = read(path, index=index)
    return frame

In [ ]:
# converts DFT trajectories -> .xyz files for MLIP runner scripts to read
# writes .xyz output files to hold model results later
confirm("Do you want to overwrite existing DFT trajectory files?")

for run in RUNS:
    write(f"dft-trajectories/{run["ads"]}_{run["slab"]}_{run["config_id"]}_dft.xyz", load_frames(run["path"]))
    
    # write initial DFT frame to MLIP trajectory file to give models the starting structure
    write(f"mlip-trajectories/{run["ads"]}_{run["slab"]}_{run["config_id"]}_mlip.xyz", load_frame(run["path"], 0))

In [8]:
# get bare slab & reference energies

REFS= {  # reference adsorbate energies
    "CO" :  None,
    "CO2":  None,
    "H2" :  None,
    "H2O":  None,
}

for mol in REFS:
    REFS[mol] = load_frame(f"../dft-ref-energy-data/adsorbates/{mol}_gas/vasprun.xml").get_potential_energy()

slab = load_frame("../dft-ref-energy-data/slabs/YO4PdCu/vasprun.xml")
slab_energy = slab.get_potential_energy()

#### 2. run MLIPs (MACE, MatterSim, UMA)

#### 3. Calculate adsorbate energies

In [ ]:
# calculate adsorbate energies from bare slab & reference energies

# TODO: figure out what's going on with CH3CHO
def calc_CH3CHO(total_energy):
    return total_energy - slab_energy + REFS["H2O"] - 2*REFS["CO"] - 3*REFS["H2"]

def calc_CO_CHO(total_energy):
    CHO_energy_diff = REFS["H2O"] - REFS["CO2"] - (3/2)*REFS["H2"]
    return total_energy - slab_energy - REFS["CO"] - CHO_energy_diff

def calc_COCHO(total_energy):
    return total_energy - slab_energy - 2*REFS["CO"] - (1/2)*REFS["H2"]

def calc_COOH(total_energy):
    return total_energy - slab_energy - REFS["CO2"] - (1/2)*REFS["H2"]

In [ ]:
# get bare slab and reference energies

In [ ]:
# # TODO: get site type from initial ion coordinates in POSCAR

# from pymatgen.analysis.adsorption import AdsorbateSiteFinder
# from pymatgen.io.ase import AseAtomsAdaptor

In [ ]:
# # get initial adsorbate coordinates from POSCAR for each DFT run

# def get_ads_init_coords(folderpath: str):
#     poscar = read(folderpath + "/POSCAR")
#     print(poscar)

# get_ads_init_coords("dft-data/YPdCu/CO_CHO_on_YO4PdCu/CO_CHO_4OYPdCu100_1")

Atoms(symbols='C2HCu98O6PdY', pbc=True, cell=[12.515153631, 12.515153631, 25.30973], momenta=..., constraint=FixAtoms(indices=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49]))
